**Merging Route Files**

In [4]:
import xml.etree.ElementTree as ET
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler")

INPUT_FILES = [
    BASE_DIR / r"randomtrips\candidate_routes.rou.xml",
    BASE_DIR / r"randomtrips\candidate_routes_seed42.rou.xml",
    BASE_DIR / r"randomtrips\candidate_routes_seed7.rou.xml",
    BASE_DIR / r"randomtrips\candidate_routes_seed99.rou.xml",
    BASE_DIR / r"randomtrips\candidate_routes_seed123.rou.xml",
]

OUT_DIR = BASE_DIR / "merged"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "candidate_routes_merged_dedup.rou.xml"

# ============================================================
# READ + DEDUP
# ============================================================

unique_routes = {}
total_routes_read = 0
bad_files = []

for file in INPUT_FILES:
    if not file.exists():
        print(f"WARNING: file not found, skipped: {file}")
        continue

    try:
        tree = ET.parse(file)
        root = tree.getroot()
    except Exception as e:
        print(f"ERROR parsing {file}: {e}")
        bad_files.append((file, str(e)))
        continue

    for veh in root.findall("vehicle"):
        route_elem = veh.find("route")
        if route_elem is None:
            continue

        edges = route_elem.attrib.get("edges", "").strip()
        if not edges:
            continue

        total_routes_read += 1

        if edges not in unique_routes:
            unique_routes[edges] = {
                "type": veh.attrib.get("type", "trafficMix")
            }

print(f"\nTotal vehicle routes read: {total_routes_read}")
print(f"Unique routes kept: {len(unique_routes)}")

if bad_files:
    print("\nBad files skipped:")
    for f, err in bad_files:
        print(f" - {f}")
        print(f"   {err}")

# ============================================================
# WRITE OUTPUT
# ============================================================

routes_root = ET.Element("routes")

for i, edges in enumerate(sorted(unique_routes.keys())):
    veh = ET.SubElement(routes_root, "vehicle", {
        "id": f"cand_{i}",
        "depart": f"{i * 0.10:.2f}",
        "type": unique_routes[edges]["type"]
    })
    ET.SubElement(veh, "route", {"edges": edges})

tree_out = ET.ElementTree(routes_root)
ET.indent(tree_out, space="    ", level=0)
tree_out.write(OUT_FILE, encoding="utf-8", xml_declaration=True)

print(f"\nMerged + deduplicated route file saved to:\n{OUT_FILE}")


Total vehicle routes read: 804150
Unique routes kept: 173

Merged + deduplicated route file saved to:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\detector_data_aktuell\new_routeSampler\merged\candidate_routes_merged_dedup.rou.xml
